# Example Deep RL agent with Keras

In [1]:
from gym import Env
from gym.spaces import Discrete, Box
import numpy as np
import random
from SimEnv import *

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import Adam

from rl.agents import DQNAgent
from rl.policy import BoltzmannQPolicy
from rl.memory import SequentialMemory

In [2]:
# Replicate runs
TEST_EPISODES = 10

# SIM PARAMETERS
QoS = 5
COST = 13 # calculate it by formula|
SLA = 1 # fit this better
VM_TYPE1 = 8
VM_TYPE2 = 5
VM_TYPE3 = 3
ALPHA = 0.5
BETA = 0.5
GAMMA = 0.97
TIMEUNITS = 60

env = SimEnv(
            qos = QoS,
            cost = COST,
            sla = SLA,
            vm_1 = VM_TYPE1,
            vm_2 = VM_TYPE2,
            vm_3 = VM_TYPE3,
            alpha = ALPHA,
            beta = BETA,
            gamma = GAMMA,
            timeunits = TIMEUNITS)

C:\Users\domin\anaconda3\lib\site-packages\gym\spaces\box.py:73: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(


In [3]:
def build_model(states, actions):
    model = Sequential()    
    model.add(Dense(24, activation='relu', input_shape=states))
    model.add(Dense(24, activation='relu'))
    model.add(Dense(actions, activation='linear'))
    return model

def build_agent(model, actions):
    policy = BoltzmannQPolicy()
    memory = SequentialMemory(limit=50000, window_length=1)
    dqn = DQNAgent(model=model, memory=memory, policy=policy, 
                  nb_actions=actions, nb_steps_warmup=10, target_model_update=1e-2)
    return dqn

In [4]:
states = env.observation_space.shape
actions = env.action_space.n

In [5]:
model = build_model(states, actions)

In [6]:
model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
dense (Dense)                (None, 24)                48        
_________________________________________________________________
dense_1 (Dense)              (None, 24)                600       
_________________________________________________________________
dense_2 (Dense)              (None, 7)                 175       
Total params: 823
Trainable params: 823
Non-trainable params: 0
_________________________________________________________________


In [7]:
dqn = build_agent(model, actions)
dqn.compile(Adam(lr=1e-3), metrics=['mae'])
dqn.fit(env, nb_steps=50000, visualize=False, verbose=1)

Training for 50000 steps ...
Interval 1 (0 steps performed)
Instructions for updating:
This property should not be used in TensorFlow 2.0, as updates are applied automatically.
    1/10000 [..............................] - ETA: 20:06 - reward: 2.0000

C:\Users\domin\anaconda3\lib\site-packages\rl\memory.py:37: UserWarning: Not enough entries to sample without replacement. Consider increasing your warm-up phase to avoid oversampling!
  warnings.warn('Not enough entries to sample without replacement. Consider increasing your warm-up phase to avoid oversampling!')


   30/10000 [..............................] - ETA: 9:49 - reward: -3.0000 

ZeroDivisionError: float division by zero

In [ ]:
scores = dqn.test(env, nb_episodes=100, visualize=False)
print(np.mean(scores.history['episode_reward']))